# 05 — Chronic Condition Burden Analysis

Analyzes the relationship between chronic condition comorbidity and healthcare utilization/costs.

Focuses on: multi-condition patterns, spending escalation with comorbidity, and condition correlations.

Requires `cms_data.db` from `01_setup.ipynb`.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DB = Path("../cms_data.db")
conn = sqlite3.connect(DB)

---
## 1. Spending escalation with number of chronic conditions

In [3]:
bene = pd.read_sql("""
SELECT
  DESYNPUF_ID,
  YEAR,
  MEDREIMB_IP + MEDREIMB_OP + MEDREIMB_CAR as total_spend,
  SP_ALZHDMTA, SP_CHF, SP_CHRNKIDN, SP_CNCR, SP_COPD, SP_DEPRESSN,
  SP_DIABETES, SP_ISCHMCHT, SP_OSTEOPRS, SP_RA_OA, SP_STRKETIA
FROM beneficiary_summary
""", conn)

# Count conditions per beneficiary (1 = has condition)
chronic_cols = ['SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN',
                'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA']

bene['num_conditions'] = (bene[chronic_cols] == 1).sum(axis=1)

# Spending by condition burden
condition_burden = bene.groupby('num_conditions').agg(
    num_beneficiaries=('DESYNPUF_ID', 'count'),
    avg_spend=('total_spend', 'mean'),
    median_spend=('total_spend', 'median'),
    max_spend=('total_spend', 'max'),
    pct_90=('total_spend', lambda x: x.quantile(0.90))
).reset_index()

condition_burden = condition_burden.round(0)

print("Spending by Number of Chronic Conditions")
print("="*90)
display(condition_burden)
condition_burden.to_csv("../data/exports/spending_by_condition_burden.csv", index=False)

Spending by Number of Chronic Conditions


,num_conditions,num_beneficiaries,avg_spend,median_spend,max_spend,pct_90
0,0,125983,268.0,0.0,73000.0,740.0
1,1,44306,1609.0,870.0,83120.0,3060.0
2,2,41353,2632.0,1430.0,91700.0,5120.0
3,3,36871,3798.0,1990.0,88490.0,8200.0
4,4,31253,5392.0,2790.0,126990.0,12290.0
5,5,24639,7395.0,3840.0,159880.0,17244.0
6,6,17785,10274.0,5510.0,147160.0,23706.0
7,7,11688,13658.0,7780.0,154970.0,31988.0
8,8,6366,17816.0,11220.0,170190.0,41480.0
9,9,2613,22220.0,14820.0,161740.0,51072.0


---
## 2. Condition co-occurrence (comorbidity patterns)

In [4]:
# For beneficiaries with each condition, what % also have other conditions?
comorbidity = pd.DataFrame()

for condition in chronic_cols:
    with_cond = bene[bene[condition] == 1]
    if len(with_cond) == 0:
        continue
    
    row = {'primary_condition': condition}
    for other_cond in chronic_cols:
        if condition != other_cond:
            pct_also_have = (with_cond[other_cond] == 1).mean() * 100
            row[f"{other_cond}_pct"] = round(pct_also_have, 1)
    
    comorbidity = pd.concat([comorbidity, pd.DataFrame([row])], ignore_index=True)

print("\nComorbidity Patterns: If patient has Condition X, likelihood of also having Y")
print("(showing top 3 co-conditions for each primary condition)")
print("="*90)

for idx, row in comorbidity.iterrows():
    primary = row['primary_condition']
    co_cols = [c for c in row.index if c.endswith('_pct')]
    co_values = [(c.replace('_pct', ''), row[c]) for c in co_cols if pd.notna(row[c])]
    co_values.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n{primary}:")
    for cond, pct in co_values[:3]:
        print(f"  + {cond}: {pct:.1f}%")


Comorbidity Patterns: If patient has Condition X, likelihood of also having Y
(showing top 3 co-conditions for each primary condition)

SP_ALZHDMTA:
  + SP_ISCHMCHT: 72.1%
  + SP_DIABETES: 66.9%
  + SP_CHF: 57.1%

SP_CHF:
  + SP_ISCHMCHT: 75.7%
  + SP_DIABETES: 67.4%
  + SP_CHRNKIDN: 39.0%

SP_CHRNKIDN:
  + SP_ISCHMCHT: 80.7%
  + SP_DIABETES: 78.5%
  + SP_CHF: 68.4%

SP_CNCR:
  + SP_ISCHMCHT: 74.7%
  + SP_DIABETES: 66.6%
  + SP_CHF: 55.8%

SP_COPD:
  + SP_ISCHMCHT: 81.9%
  + SP_DIABETES: 75.6%
  + SP_CHF: 69.2%

SP_DEPRESSN:
  + SP_ISCHMCHT: 70.4%
  + SP_DIABETES: 65.9%
  + SP_CHF: 53.7%

SP_DIABETES:
  + SP_ISCHMCHT: 73.5%
  + SP_CHF: 55.0%
  + SP_DEPRESSN: 38.5%

SP_ISCHMCHT:
  + SP_DIABETES: 62.9%
  + SP_CHF: 52.9%
  + SP_DEPRESSN: 35.2%

SP_OSTEOPRS:
  + SP_ISCHMCHT: 69.3%
  + SP_DIABETES: 63.8%
  + SP_CHF: 50.4%

SP_RA_OA:
  + SP_ISCHMCHT: 76.2%
  + SP_DIABETES: 71.4%
  + SP_CHF: 56.2%

SP_STRKETIA:
  + SP_ISCHMCHT: 82.8%
  + SP_DIABETES: 76.7%
  + SP_CHF: 70.5%


---
## 3. High-risk comorbidity combinations

In [5]:
# Spending for specific dual-condition combinations
dual_combinations = pd.DataFrame()

combinations = [
    ('SP_DIABETES', 'SP_CHF'),
    ('SP_DIABETES', 'SP_ISCHMCHT'),
    ('SP_CHF', 'SP_COPD'),
    ('SP_ISCHMCHT', 'SP_DEPRESSN'),
    ('SP_CHRNKIDN', 'SP_DIABETES'),
]

for cond1, cond2 in combinations:
    both = bene[(bene[cond1] == 1) & (bene[cond2] == 1)]
    either = bene[((bene[cond1] == 1) | (bene[cond2] == 1))]
    neither = bene[(bene[cond1] == 2) & (bene[cond2] == 2)]
    
    row = {
        'combination': f"{cond1.replace('SP_', '')} + {cond2.replace('SP_', '')}",
        'both_conditions_avg_spend': round(both['total_spend'].mean()) if len(both) > 0 else None,
        'either_condition_avg_spend': round(either['total_spend'].mean()) if len(either) > 0 else None,
        'neither_condition_avg_spend': round(neither['total_spend'].mean()) if len(neither) > 0 else None,
        'prevalence_both_pct': round((len(both) / len(bene) * 100), 1)
    }
    dual_combinations = pd.concat([dual_combinations, pd.DataFrame([row])], ignore_index=True)

print("\nSpending Impact of Dual-Condition Combinations")
print("="*90)
display(dual_combinations)
dual_combinations.to_csv("../data/exports/dual_condition_combinations.csv", index=False)


Spending Impact of Dual-Condition Combinations


,combination,both_conditions_avg_spend,either_condition_avg_spend,neither_condition_avg_spend,prevalence_both_pct
0,DIABETES + CHF,9810,6683,1003,20.0
1,DIABETES + ISCHMCHT,8622,6253,750,26.7
2,CHF + COPD,13350,7859,1472,8.8
3,ISCHMCHT + DEPRESSN,9286,6395,974,14.9
4,CHRNKIDN + DIABETES,12495,7337,1139,13.3


---
## 4. Condition progression by year

In [6]:
# For each condition, what % of beneficiaries developed it year-over-year?
bene_pivot = bene.pivot_table(
    index='DESYNPUF_ID',
    columns='YEAR',
    values=chronic_cols[0],
    aggfunc='first'
)

progression = pd.DataFrame()

for condition in chronic_cols:
    data_2008 = bene[bene['YEAR'] == 2008][condition].value_counts()
    data_2009 = bene[bene['YEAR'] == 2009][condition].value_counts()
    data_2010 = bene[bene['YEAR'] == 2010][condition].value_counts()
    
    pct_2008 = (data_2008.get(1, 0) / len(bene[bene['YEAR'] == 2008]) * 100).round(1) if len(bene[bene['YEAR'] == 2008]) > 0 else 0
    pct_2009 = (data_2009.get(1, 0) / len(bene[bene['YEAR'] == 2009]) * 100).round(1) if len(bene[bene['YEAR'] == 2009]) > 0 else 0
    pct_2010 = (data_2010.get(1, 0) / len(bene[bene['YEAR'] == 2010]) * 100).round(1) if len(bene[bene['YEAR'] == 2010]) > 0 else 0
    
    row = {
        'condition': condition,
        'prevalence_2008': pct_2008,
        'prevalence_2009': pct_2009,
        'prevalence_2010': pct_2010,
        'change_2008_to_2010': (pct_2010 - pct_2008).round(1)
    }
    progression = pd.concat([progression, pd.DataFrame([row])], ignore_index=True)

print("\nCondition Prevalence Over Time (2008–2010)")
print("="*90)
display(progression)
progression.to_csv("../data/exports/condition_progression_by_year.csv", index=False)


Condition Prevalence Over Time (2008–2010)


,condition,prevalence_2008,prevalence_2009,prevalence_2010,change_2008_to_2010
0,SP_ALZHDMTA,19.3,23.1,16.6,-2.7
1,SP_CHF,28.5,34.4,25.9,-2.6
2,SP_CHRNKIDN,16.1,20.7,13.9,-2.2
3,SP_CNCR,6.4,8.2,5.1,-1.3
4,SP_COPD,13.5,15.6,9.0,-4.5
5,SP_DEPRESSN,21.3,24.5,17.8,-3.5
6,SP_DIABETES,37.9,41.7,29.2,-8.7
7,SP_ISCHMCHT,42.1,47.7,37.4,-4.7
8,SP_OSTEOPRS,17.3,19.1,13.0,-4.3
9,SP_RA_OA,15.4,17.5,9.7,-5.7


In [ ]:
conn.close()
print("\nChronic condition analysis complete. Outputs saved to data/exports/")